# GeometricNearestNeighbors usage examples

Anton Antonov   
PythonForPrediction at WordPress   
June 2026

---

## Introduction

This document ([notebook](https://github.com/antononcube/PythonForPrediction-blog/blob/main/Notebooks/Jupyter/Geometric-nearest-neighbors-processing.ipynb)) presents a computational approach to identifying outliers in a set of multidimensional (geometric) points. The method discussed and demonstrated is based on nearest-neighbor statistics. Related workflow extensions, such as visualization and classification, are also shown.

The Python package ["GeometricNearestNeighborsProcessor"](https://pypi.org/project/GeometricNearestNeighborsProcessor/) [AAp1] is used (which, as the name indicates, was specifically developed for tackling these kinds of problems.)

----

## Purpose and theoretical background

Consider the following computational tasks for a given set of $n$-dimensional (nD) points $P$:

1. Find the points of $P$ that are outliers or anomalies
2. Find the points of another set $P_1$ that can be seen as anomalies wrt to $P$
3. For a given nD point $s$ find its Nearest Neighbors (NNs) in $P$
   - The points of $P$ can have labels 
   - It might be desired to get the distances and labels of the NNs of $s$  
4. Plot the points $P$ with minimal setup or specification writing
5. Give the (sparse) proximity matrix of $P$ for a specified number of neighbors 

Let us define an anomalous point as one that is "too far" from the other points.  
Which points are "too far" from the rest can be determined by examining statistics of distances between each point and $k$ nearest neighbors of it.

More concretely, point anomalies are found in the following way:

1. Input:
   - Points `P` as a data frame, list, or dictionary
   - Number of nearest neighbors `n`
   - Distance function `d`
   - Aggregation function `a`
2. For each point of `P` 
   - Find its `n` nearest neighbors
   - Aggregate with `a` the corresponding `n` distances
3. Using the statistics from the previous step -- a 1D array -- find outlier identification parameters
   - Like, Hampel-, SPLUS-, or Quartile parameters
4. Identify anomalies using the parameters of the previous step


---

## Setup

Here are imported (and aliased) the packages used below:

In [17]:
from GeometricNearestNeighborsProcessor import *
from RandomDataGenerators import *
from OutlierIdentifiers import *

import numpy
import random

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

---

## Usage examples

Generate random points (using the package ["RandomDataGenerators"](https://github.com/antononcube/Python-packages/tree/main/RandomDataGenerators), [AAp2]):

In [18]:
random.seed(23)
dfPoints = random_data_frame(
    n_rows=30, 
    columns_spec = ["X", "Y"], 
    generators= {"X": numpy.random.normal, "Y": numpy.random.normal}
)

print(dfPoints.shape)
print(dfPoints[1:6])

(30, 2)
          X         Y
1  0.033569  0.564870
2 -2.092372 -1.821958
3  0.414600  0.198779
4  1.425464 -1.009210
5 -1.451830  0.529209


Here is a summary of the point coordinates:

In [19]:
dfPoints.describe()

,X,Y
count,30.000000,30.000000
mean,-0.231960,0.068224
std,1.022492,0.958301
min,-2.420255,-2.202532
25%,-0.732376,-0.543855
50%,-0.256209,0.164172
75%,0.405636,0.724696
max,1.577077,2.147675


Here is a plot of the points:

In [20]:
fig = px.scatter(dfPoints, x="X", y="Y", template="plotly_dark")
fig.show()

Here is a typical pipeline of geometric nearest neighbors processing:

1. Initiate the Geometric Nearest Neighbors (GNNs) object with a set of points
2. Create the nearest-neighbor finder (a function or a function-like object)
3. Compute the outlier detection thresholds using:
   - `10` nearest neighbors
   - The mean of the radiuses as a statistic
   - The outlier detector "Quartile"
4. Find anomalies (among the points of the GNN object)
5. Echo the found anomalies


In [21]:
gnnObj = (GeometricNearestNeighborsProcessor(dfPoints)
   .make_nearest_function(distance_function = "EuclideanDistance")
   .compute_thresholds(number_of_nearest_neighbors = 10, aggregation_function = "mean", outlier_identifier = "QuartileIdentifierParameters")
   .find_anomalies()
   .echo_function_value("Anomaly points:")
)

Anomaly points:
          X         Y    Radius
0 -2.092372 -1.821958  1.867283
1  1.031366 -2.202532  1.740659
2 -2.420255  0.366653  1.433153


One way to plot the anomalies together with the data points using the package "plotly" is to merge into one data frame and add an indicator column:

In [22]:
# Assign data
df1 = gnnObj.take_data()
df2 = gnnObj.take_value()

# Mark points that are in df2
df1['highlight'] = df1.merge(df2.assign(flag=1), on=['X', 'Y'], how='left')['flag'].fillna(0)

# Convert to category for plotting
df1['highlight'] = df1['highlight'].map({0: 'data', 1: 'anomalies'})

# Plot
fig = px.scatter(df1, 
    title="Random data", 
    x='X', y='Y', color='highlight',
    color_discrete_map={'data': 'blue', 'anomalies': 'red'},
    template="plotly_dark")

fig.show()

Alternatively, the anomaly points can be shown as smaller, in red, and on top of the data points:

In [23]:
fig = go.Figure()

# All points (larger)
fig.add_trace(go.Scatter(
    x=df1['X'],
    y=df1['Y'],
    mode='markers',
    marker=dict(size=12, color='blue'),
    name='data'
))

# Anomaly points (smaller, on top)
fig.add_trace(go.Scatter(
    x=df2['X'],
    y=df2['Y'],
    mode='markers',
    marker=dict(size=6, color='red', line=dict(width=1, color='black')),
    name='anomalies'
))

fig.update_layout(
    title="Random data", 
    template="plotly_dark",
    xaxis_title="X Axis",
    yaxis_title="Y Axis"
)

fig.show()

The following data can be used :

```python
aPoints = [(r["X"], r["Y"]) for r in dfPoints.to_dict(orient="records")]
aPoints = dict(zip(random_word(len(aPoints)), aPoints))
```

With that the plots above have to use the columns "x1" and "x2". Or the attribute "data" of `gnnObj` have to be changed to have the columns "X" and "Y".

----

## Nearest neighbors

Here we generate another set of random points using the same random point generators:

In [24]:
dfPoints2 = random_data_frame(n_rows=40, columns_spec = ["X", "Y"], generators= {"X": numpy.random.normal, "Y": numpy.random.normal})
print(dfPoints2.shape)

(40, 2)


For each point of the second set find its top-3 nearest neighbors in the first set:

In [25]:
dfNNs = pd.concat(
    [
        gnnObj.find_nearest(point=[row["X"], row["Y"]], n=3)
        .take_value()
        .assign(SearchIndex=i)
        for i, row in enumerate(dfPoints2.to_dict(orient="records"))
    ],
    ignore_index=True
)

dfNNs

,Index,ID,Distance,SearchIndex
0,23,p23,0.536986,0
1,12,p12,1.099262,0
2,5,p05,1.514127,0
3,6,p06,0.519515,1
4,0,p00,0.685185,1
...,...,...,...,...
115,18,p18,0.744924,38
116,15,p15,0.750928,38
117,0,p00,0.452745,39
118,10,p10,0.492998,39


----

## Classification

Here the points of second set are classified into being anomalous or not:

In [26]:
gnnObj.classify(dfPoints2).take_value()

,Index,Radius,Label
0,0,1.902066,False
1,1,0.763979,True
2,2,0.949404,True
3,3,1.011849,True
4,4,0.566731,True
5,5,0.546325,True
6,6,1.110571,True
7,7,0.801986,True
8,8,0.490496,True
9,9,0.568235,True


Plot the original data points (in gray) and the new data points by marking the anomalous ones:

In [27]:
df0 = gnnObj.take_data()

dfClassified = gnnObj.classify(dfPoints2).take_value()
dfCombined = dfPoints2.join(dfClassified)

df1 = dfCombined[dfCombined["Label"] == True]
df2 = dfCombined[dfCombined["Label"] == False]

fig = go.Figure()

# Original data points
fig.add_trace(go.Scatter(
    x=df0['X'],
    y=df0['Y'],
    mode='markers',
    marker=dict(size=8, color='gray'),
    name='data'
))

# Non-anomaly points
fig.add_trace(go.Scatter(
    x=df1['X'],
    y=df1['Y'],
    mode='markers',
    marker=dict(size=10, color='blue', line=dict(width=1, color='black')),
    name='non-anomalies'
))

# Anomaly points
fig.add_trace(go.Scatter(
    x=df2['X'],
    y=df2['Y'],
    mode='markers',
    marker=dict(size=10, color='red', line=dict(width=1, color='black')),
    name='anomalies'
))

fig.update_layout(
    title = "Anomaly classification results",
    template="plotly_dark",
    xaxis_title="X Axis",
    yaxis_title="Y Axis"
)

fig.show()

----

## Proximity matrix

For some of the algorithms for recommendations systems or search engines it is helpful to have a metrics of the distances from each point to its top-K nearest neighbors. We call that matrix "proximity matrix". Here is the proximity matrix of the GNN object created (and trained) above:

In [28]:
mat=gnnObj.compute_proximity_matrix().take_value()
mat

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 330 stored elements and shape (30, 30)>

Here is a corresponding matrix plot:

In [29]:
rows, cols = mat.nonzero()

fig = go.Figure(data=go.Scattergl(
    x=cols,
    y=rows,
    mode='markers',
    marker=dict(size=3)
))

fig.update_layout(
    title="Proximity matrix",
    xaxis_title="Columns",
    yaxis_title="Rows",
    yaxis_autorange='reversed',
    xaxis=dict(scaleanchor='y', scaleratio=1),
    yaxis=dict(constrain='domain'),
    width=600,
    height=600,
    template="plotly_dark"
)

fig.show()

----

## Additional comments


- The package ["GeometricNearestNeighborsProcessor"](https://pypi.org/project/GeometricNearestNeighborsProcessor) provides the class `GeometricNearestNeighborsProcessor` that can be used to construct chainable, monadic pipeline-like behavior.

- Plotting of the data points is done via ["plotly"](https://plotly.com/python/) -- just a scatter plot of 2D points for now. 

- The chainable behavior of the methods of the class `GeometricNearestNeighborsProcessor`is implemented 
  by following the principle that all methods return `self`, except the so-called "takers".
  - I.e., methods with names that start with "take_".

- The core Nearest Neighbors (NNs) finding functionality is provided by the ["scikit-learn"](scikit-learn.org) class [NearestNeighbors](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.NearestNeighbors.html).

- The NNs finding algorithms used by `GeometricNearestNeighborsProcessor` are "scan" and "kdtree".
  - "scan" is implemented in the class `GeometricNearestNeighborsProcessor` instead of delegating to scikit-learn's `NearestNeighbors` "brute" algorithm.
  - "kdtree" delegates to the "kd_tree" algorithm of `NearestNeighbors`.


---

## References

### Python

[AAp1] Anton Antonov, [GeometricNearestNeighborsProcessor](https://github.com/antononcube/Python-GeometricNearestNeighborsProcessor), Python package, (2026), [GitHub/antononcube](https://github.com/antononcube).([PIPy.org](https://pypi.org/project/GeometricNearestNeighborsProcessor/).)

[AAp2] Anton Antonov, [RandomDataGenerators](https://github.com/antononcube/Python-packages/tree/main/RandomDataGenerators), Python package, (2021-2026), [GitHub/antononcube](https://github.com/antononcube). ([PIPy.org](https://pypi.org/project/RandomDataGenerators/).)

[AAp3] Anton Antonov, [OutlierIdentifiers](https://github.com/antononcube/Python-packages/tree/main/OutlierIdentifiers), Python package, (2024), [GitHub/antononcube](https://github.com/antononcube). ([PIPy.org](https://pypi.org/project/OutlierIdentifiers/).)

### R

[AAp4] Anton Antonov, [OutlierIdentifiers](https://github.com/antononcube/R-packages/tree/master/OutlierIdentifiers), R package, (2019-2024), [GitHub/antononcube](https://github.com/antononcube).

[AAp5] Anton Antonov, [GNNMon-R](https://github.com/antononcube/R-packages/tree/master/GNNMon-R), R package, (2019-2025), [GitHub/antononcube](https://github.com/antononcube).

[AAp6] Anton Antonov, [KDTreeAlgorithm](https://github.com/antononcube/R-packages/tree/master/KDTreeAlgorithm), R package, (2025), [GitHub/antononcube](https://github.com/antononcube).

### Wolfram Language

[AAp7] Anton Antonov, [MonadicGeometricNearestNeighbors](https://resources.wolframcloud.com/PacletRepository/resources/AntonAntonov/MonadicGeometricNearestNeighbors), Wolfram Language paclet, (2023-2025), [Wolfram Language Paclet Repository](https://resources.wolframcloud.com/PacletRepository).

[AAp8] Anton Antonov, [OutlierIdentifiers](https://resources.wolframcloud.com/PacletRepository/resources/AntonAntonov/OutlierIdentifiers/), Wolfram Language paclet, (2023), [Wolfram Language Paclet Repository](https://resources.wolframcloud.com/PacletRepository).
